# 🚀 EDA — Space Missions Dataset
**Análisis Exploratorio de Datos · CEIA 2026**  
Dataset: *Space Missions (1957–2022)* — Kaggle  

| | |
|---|---|
| **Registros** | 4 630 |
| **Variables** | 9 |
| **Período** | 1957 – 2022 |


## 1 · Importaciones

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print(f"pandas     {pd.__version__}")
print(f"numpy      {np.__version__}")
print(f"matplotlib {plt.matplotlib.__version__}")
print(f"seaborn    {sns.__version__}")


## 2 · Carga de datos desde GitHub

In [2]:
URL = "https://raw.githubusercontent.com/cimberlina/CEIA2026_AdD/main/space_missions.csv"

df = pd.read_csv(URL, encoding='latin-1')
df['Price'] = pd.to_numeric(df['Price'].astype(str).str.replace(',', ''), errors='coerce')
df['Date']  = pd.to_datetime(df['Date'], errors='coerce')
df['Year']  = df['Date'].dt.year

print(f"Shape: {df.shape}")
df.head()


## 3 · Tipos de variables

In [3]:
print("─── dtypes ───────────────────────────────")
print(df.dtypes)
print()
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include='number').columns.tolist()
dt_cols  = df.select_dtypes(include='datetime').columns.tolist()
print(f"Categóricas  ({len(cat_cols)}): {cat_cols}")
print(f"Numéricas    ({len(num_cols)}): {num_cols}")
print(f"Fecha/hora   ({len(dt_cols)}): {dt_cols}")


## 4 · Estadística descriptiva
### 4.1 Variables numéricas

In [4]:
df.describe()

### 4.2 Variables categóricas (top valores)

In [5]:
for col in ['Company', 'RocketStatus', 'MissionStatus']:
    print(f"\n{'─'*40}")
    print(f"  {col}  (únicos: {df[col].nunique()})")
    print(df[col].value_counts().head(5).to_string())


## 5 · Duplicados y valores nulos

In [6]:
print(f"Filas duplicadas: {df.duplicated().sum()}")
print()
nulls = df.isnull().sum()
pct   = (nulls / len(df) * 100).round(2)
print(pd.DataFrame({'Nulos': nulls, '%': pct}).to_string())


## 6 · Detección de outliers — Variable `Price`

In [7]:
price = df['Price'].dropna()
Q1, Q3 = price.quantile(0.25), price.quantile(0.75)
IQR    = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outliers = df[(df['Price'] < lower) | (df['Price'] > upper)]

print(f"Q1 = {Q1:.2f} M USD   Q3 = {Q3:.2f} M USD   IQR = {IQR:.2f}")
print(f"Límite inferior : {lower:.2f} M USD")
print(f"Límite superior : {upper:.2f} M USD")
print(f"Outliers detectados: {len(outliers)} ({len(outliers)/len(price)*100:.1f}% del precio conocido)")
print()
print("Top 5 lanzamientos más costosos:")
print(df.nlargest(5, 'Price')[['Company','Mission','Year','Price']].to_string(index=False))


## 7 · Visualizaciones

In [8]:
PALETTE = {
    'bg':      '#0d1117',
    'panel':   '#161b22',
    'accent1': '#58a6ff',
    'accent2': '#f78166',
    'accent3': '#3fb950',
    'accent4': '#d2a8ff',
    'text':    '#e6edf3',
    'grid':    '#30363d',
}

plt.rcParams.update({
    'figure.facecolor':  PALETTE['bg'],
    'axes.facecolor':    PALETTE['panel'],
    'axes.edgecolor':    PALETTE['grid'],
    'axes.labelcolor':   PALETTE['text'],
    'xtick.color':       PALETTE['text'],
    'ytick.color':       PALETTE['text'],
    'text.color':        PALETTE['text'],
    'grid.color':        PALETTE['grid'],
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'font.size':         11,
})
print("Paleta configurada ✓")


### 📈 Gráfico 1 — Lanzamientos por año y tasa de éxito

In [9]:
yearly = df.groupby('Year').agg(
    total=('MissionStatus', 'count'),
    success=('MissionStatus', lambda x: (x == 'Success').sum())
).reset_index()
yearly['success_rate'] = yearly['success'] / yearly['total'] * 100

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(yearly['Year'], yearly['total'],
        color=PALETTE['accent1'], alpha=0.75, width=0.85)
ax1.set_ylabel('Nº de lanzamientos', color=PALETTE['accent1'])
ax1.tick_params(axis='y', colors=PALETTE['accent1'])
ax1.set_xlabel('Año')
ax1.set_xlim(1956.5, 2022.5)

ax2 = ax1.twinx()
ax2.plot(yearly['Year'], yearly['success_rate'],
         color=PALETTE['accent3'], lw=2.2, marker='o', markersize=3)
ax2.set_ylabel('Tasa de éxito (%)', color=PALETTE['accent3'])
ax2.tick_params(axis='y', colors=PALETTE['accent3'])
ax2.set_ylim(0, 110)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())

ax1.grid(axis='y', alpha=0.3)
handles = [
    plt.Rectangle((0,0),1,1, color=PALETTE['accent1'], alpha=0.75),
    plt.Line2D([0],[0], color=PALETTE['accent3'], lw=2, marker='o')
]
ax1.legend(handles, ['Lanzamientos totales', 'Tasa de éxito (%)'],
           loc='upper left', framealpha=0.2,
           labelcolor=PALETTE['text'], facecolor=PALETTE['panel'])
ax1.set_title('Lanzamientos espaciales por año y tasa de éxito (1957–2022)',
              pad=12, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 📊 Gráfico 2 — Top 10 compañías y estado de misión

In [10]:
top10         = df['Company'].value_counts().head(10)
status_counts = df['MissionStatus'].value_counts()
colors_pie    = [PALETTE['accent3'], PALETTE['accent2'], PALETTE['accent1'], PALETTE['accent4']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.barh(top10.index[::-1], top10.values[::-1],
         color=PALETTE['accent1'], edgecolor='none', height=0.7, alpha=0.85)
for i, val in enumerate(top10.values[::-1]):
    ax1.text(val + 10, i, f'{val:,}', va='center', fontsize=9, color=PALETTE['text'])
ax1.set_title('Top 10 Compañías por nº de lanzamientos', fontsize=12, fontweight='bold')
ax1.set_xlabel('Lanzamientos')
ax1.set_xlim(0, top10.max() * 1.18)
ax1.grid(axis='x', alpha=0.3)

wedges, texts, autotexts = ax2.pie(
    status_counts, labels=status_counts.index,
    autopct='%1.1f%%', startangle=140,
    colors=colors_pie, explode=[0.04]*len(status_counts),
    pctdistance=0.78,
    wedgeprops=dict(edgecolor=PALETTE['bg'], linewidth=1.5))
for t  in texts:     t.set_color(PALETTE['text'])
for at in autotexts: at.set_color(PALETTE['bg']); at.set_fontsize(9)
ax2.set_facecolor(PALETTE['bg'])
ax2.set_title('Distribución del estado de las misiones', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


### 💰 Gráfico 3a — Distribución del costo de lanzamiento

In [11]:
df_price = df.dropna(subset=['Price'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(df_price['Price'], bins=40, color=PALETTE['accent1'],
        alpha=0.7, edgecolor='none', density=True)
df_price['Price'].plot.kde(ax=ax, color=PALETTE['accent2'], lw=2.5)
ax.set_title('Distribución del costo de lanzamiento (USD M)', fontsize=13, fontweight='bold')
ax.set_xlabel('Precio (USD M)')
ax.set_ylabel('Densidad')
ax.grid(alpha=0.3)

for q, lbl, col in [(df_price['Price'].quantile(0.25), 'Q1',      PALETTE['accent3']),
                    (df_price['Price'].median(),        'Mediana', '#ffd700'),
                    (df_price['Price'].quantile(0.75),  'Q3',      PALETTE['accent4'])]:
    ax.axvline(q, color=col, lw=1.8, ls='--')
    ax.text(q + 15, ax.get_ylim()[1] * 0.88, lbl, color=col, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


### 📦 Gráfico 3b — Precio por estado de misión (boxplot)

In [12]:
status_order  = ['Success', 'Failure', 'Partial Failure', 'Prelaunch Failure']
status_colors = [PALETTE['accent3'], PALETTE['accent2'], PALETTE['accent1'], PALETTE['accent4']]
bp_data       = [df_price[df_price['MissionStatus'] == s]['Price'].dropna() for s in status_order]

fig, ax = plt.subplots(figsize=(12, 6))
bp = ax.boxplot(bp_data, patch_artist=True, widths=0.5,
                medianprops=dict(color='#ffd700', lw=2.5),
                flierprops=dict(marker='o', markerfacecolor=PALETTE['grid'],
                                markersize=4, alpha=0.5),
                whiskerprops=dict(color=PALETTE['text'], lw=1.5),
                capprops=dict(color=PALETTE['text'], lw=1.5))
for patch, color in zip(bp['boxes'], status_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xticks(range(1, len(status_order) + 1))
ax.set_xticklabels(status_order, fontsize=11)
ax.set_title('Distribución del precio por estado de misión', fontsize=13, fontweight='bold')
ax.set_ylabel('Precio (USD M)', fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Anotar mediana sobre cada caja
for i, data in enumerate(bp_data, 1):
    med = data.median()
    ax.text(i, med + 30, f'{med:.0f}M', ha='center', fontsize=9,
            color='#ffd700', fontweight='bold')

plt.tight_layout()
plt.show()


## 8 · Conclusiones del EDA

| Aspecto | Hallazgo |
|---|---|
| **Escala** | 4 630 lanzamientos en 65 años (1957–2022) |
| **Tasa de éxito** | 89.9 % — mejora notoria desde los años 60 |
| **Nulos críticos** | `Price` falta en 72.7 % de los registros |
| **Duplicados** | Solo 1 fila duplicada (despreciable) |
| **Outliers precio** | 164 outliers IQR; máximo: 5 000 M USD (NASA) |
| **Dominancia** | RVSN USSR concentra 38 % de todos los lanzamientos |
| **Cohetes activos** | Solo 21.8 % de los modelos siguen operativos |
| **SpaceX** | 182 misiones con 100 % de éxito registrado en el dataset |

### Próximos pasos sugeridos
- Imputar `Price` con modelos basados en compañía/año/cohete
- Ingeniería de features: país de origen a partir de `Location`
- Serie temporal sobre lanzamientos anuales
- Clasificación binaria: predecir `MissionStatus` (Success / Failure)
